In [1]:
import pandas as pd
import numpy as np
%matplotlib widget
import matplotlib.pyplot as plt
import os
import sys
sys.path.append('/home/wolfgang/repos/sleep_research_io')
from sleep_research_functions import load_sleep_data, write_to_hdf5_file, get_metadata
from tqdm import tqdm

In [2]:
def sleep_indices(stages, verbose=False):

    stages = stages.dropna()

    hours_data_available = stages.shape[0]/fs/3600
    hours_sleep = sum(stages<5)/fs/3600

    if hours_sleep == 0:
        return [hours_data_available, hours_sleep, 
               np.nan, np.nan, np.nan, np.nan, 
               np.nan, np.nan, np.nan]

    perc_n1 = np.round((sum(stages==3)/fs/3600) / hours_sleep * 100, 0).astype('int64')
    perc_n2 = np.round((sum(stages==2)/fs/3600) / hours_sleep * 100, 0).astype('int64')
    perc_n3 = np.round((sum(stages==1)/fs/3600) / hours_sleep * 100, 0).astype('int64')
    perc_r  = np.round((sum(stages==4)/fs/3600) / hours_sleep * 100, 0).astype('int64')

    # sleep fragmentation index:
    w_or_n1 = np.isin(stages,[5,3])
    deep_sleep = np.isin(stages,[1,2,4])
    fragmentation_shift = np.logical_and(w_or_n1[1:], deep_sleep[:-1])
    fragmentation_pos = np.where(fragmentation_shift)[0]
#     fragmentation_pos = smooth_fragmentation_index(fragmentation_pos)
    sfi = np.round(len(fragmentation_pos)/hours_sleep, 1)

    # SFI2, based only on transitions to W from N2, N3 or R
    stages_without_n1 = stages[np.isin(stages,[1,2,4,5])]
    w = np.isin(stages_without_n1,[5])
    deep_sleep = np.isin(stages_without_n1,[1,2,4])
    fragmentation_shift = np.logical_and(w[1:], deep_sleep[:-1])
    fragmentation_pos = np.where(fragmentation_shift)[0]
#     fragmentation_pos = smooth_fragmentation_index(fragmentation_pos, fs=10)
    sfi2 = np.round(len(fragmentation_pos)/hours_sleep, 1)
    
    # arousal index, based  on transitions to W from sleep
    w = np.isin(stages,[5])
    sleep = np.isin(stages,[1,2,3,4])
    fragmentation_shift = np.logical_and(w[1:], sleep[:-1])
    fragmentation_pos = np.where(fragmentation_shift)[0]
#     fragmentation_pos = smooth_fragmentation_index(fragmentation_pos, fs=10)
    arousal_index = np.round(len(fragmentation_pos)/hours_sleep, 1)

    hours_sleep = np.round(hours_sleep, 10)
    hours_data_available = np.round(hours_data_available, 1)

    if verbose: 
        print(f'Hours of data available:     {hours_data_available:.1f}')
        print(f'Hours of sleep:              {hours_sleep:.10f}')
        print(f'N1, N2, N3, R in %:          {[perc_n1, perc_n2, perc_n3, perc_r]}')
        print(f'Sleep Fragmentation Index:   {sfi}')
        print(f'Sleep Fragmentation Index2:  {sfi2}')
        print(f'Arousal Index:               {arousal_index}')

    return [hours_data_available, hours_sleep, 
           perc_n1, perc_n2, perc_n3, perc_r, 
           sfi, sfi2, arousal_index]

In [3]:
def ibi_statistics(Xy, data, study_id, name_addon = ''):

    Xy.loc[Xy.study_id == study_id, 'IBI_mean' + name_addon] = data['IBI'][data.peaks==1].mean()
    Xy.loc[Xy.study_id == study_id, 'IBI_std' + name_addon] = data['IBI'][data.peaks==1].std()
    Xy.loc[Xy.study_id == study_id, 'IBI_median' + name_addon] = data['IBI'][data.peaks==1].median()
    Xy.loc[Xy.study_id == study_id, 'IBI_Q75' + name_addon] = data['IBI'][data.peaks==1].quantile(0.75)
    Xy.loc[Xy.study_id == study_id, 'IBI_Q25' + name_addon] = data['IBI'][data.peaks==1].quantile(0.25)
    Xy.loc[Xy.study_id == study_id, 'IBI_RMSSD' + name_addon] = np.sqrt(np.mean(data['IBI'][data.peaks==1].diff().values[2:]**2))
    Xy.loc[Xy.study_id == study_id, 'IBI_SDA' + name_addon] = data['IBI_mean_5min'].std()
    Xy.loc[Xy.study_id == study_id, 'IBI_ASD' + name_addon] = data['IBI_std_5min'].mean()
    
    return Xy

def rr_statistics(Xy, data, study_id, name_addon = ''):

    Xy.loc[Xy.study_id == study_id, 'RR_mean' + name_addon] = data['rr_10s_smooth'].mean()
    Xy.loc[Xy.study_id == study_id, 'RR_std' + name_addon] = data['rr_10s_smooth'].std()
    Xy.loc[Xy.study_id == study_id, 'RR_median' + name_addon] = data['rr_10s_smooth'].median()
    Xy.loc[Xy.study_id == study_id, 'RR_Q75' + name_addon] = data['rr_10s_smooth'].quantile(0.75)
    Xy.loc[Xy.study_id == study_id, 'RR_Q25' + name_addon] = data['rr_10s_smooth'].quantile(0.25)
    Xy.loc[Xy.study_id == study_id, 'RR_SDA' + name_addon] = data['rr_10s_smooth'].rolling('5min', min_periods=1).mean().std()
    Xy.loc[Xy.study_id == study_id, 'RR_ASD' + name_addon] = data['rr_10s_smooth'].rolling('5min', min_periods=1).std().mean()
    
    return Xy

def mv_statistics(Xy, data, study_id, name_addon = ''):

    Xy.loc[Xy.study_id == study_id, 'MV_mean' + name_addon] = data['ventilation_cvar_30sec'].mean()
    Xy.loc[Xy.study_id == study_id, 'MV_std' + name_addon] = data['ventilation_cvar_30sec'].std()
    Xy.loc[Xy.study_id == study_id, 'MV_median' + name_addon] = data['ventilation_cvar_30sec'].median()
    Xy.loc[Xy.study_id == study_id, 'MV_Q75' + name_addon] = data['ventilation_cvar_30sec'].quantile(0.75)
    Xy.loc[Xy.study_id == study_id, 'MV_Q25' + name_addon] = data['ventilation_cvar_30sec'].quantile(0.25)
    Xy.loc[Xy.study_id == study_id, 'MV_SDA' + name_addon] = data['ventilation_cvar_30sec'].rolling('5min', min_periods=1).mean().std()
    Xy.loc[Xy.study_id == study_id, 'MV_ASD' + name_addon] = data['ventilation_cvar_30sec'].rolling('5min', min_periods=1).std().mean()
    
    return Xy

def breathing_stability_statistics(Xy, data, study_id, name_addon = ''):

    Xy.loc[Xy.study_id == study_id, 'BSI_mean' + name_addon] = data['instability_index_30sec'].mean()
    Xy.loc[Xy.study_id == study_id, 'BSI_std' + name_addon] = data['instability_index_30sec'].std()
    Xy.loc[Xy.study_id == study_id, 'BSI_median' + name_addon] = data['instability_index_30sec'].median()
    Xy.loc[Xy.study_id == study_id, 'BSI_Q75' + name_addon] = data['instability_index_30sec'].quantile(0.75)
    Xy.loc[Xy.study_id == study_id, 'BSI_Q25' + name_addon] = data['instability_index_30sec'].quantile(0.25)
    Xy.loc[Xy.study_id == study_id, 'BSI_SDA' + name_addon] = data['instability_index_30sec'].rolling('5min', min_periods=1).mean().std()
    Xy.loc[Xy.study_id == study_id, 'BSI_ASD' + name_addon] = data['instability_index_30sec'].rolling('5min', min_periods=1).std().mean()
    
    return Xy


# DISTRIBUTION STATISTICS FOR BREATHING METRICS (during sleep):

def breathing_metrics_routine(Xy, data):

    # all of those function have the same input/out, so we can loop over them to compute the different statistics:
    functions_to_apply = [
        ibi_statistics,
        rr_statistics,
        mv_statistics,
        breathing_stability_statistics
    ]

    for function_sel in functions_to_apply:

        # Overall IBI statistics
        Xy = function_sel(Xy, data, study_id, name_addon = '')

        ### Stratified by sleep stages:

        for stage_prediction_version, stage_v in zip(stage_prediction_versions, stage_vs):

            data_sel = data.loc[np.isin(data[stage_prediction_version], [3])]
            Xy = function_sel(Xy, data_sel, study_id, name_addon = f'_{stage_v}_N1')

            data_sel = data.loc[np.isin(data[stage_prediction_version], [2])]
            Xy = function_sel(Xy, data_sel, study_id, name_addon = f'_{stage_v}_N2')

            data_sel = data.loc[np.isin(data[stage_prediction_version], [1])]
            Xy = function_sel(Xy, data_sel, study_id, name_addon = f'_{stage_v}_N3')

            data_sel = data.loc[np.isin(data[stage_prediction_version], [1,2])]
            Xy = function_sel(Xy, data_sel, study_id, name_addon = f'_{stage_v}_N2N3')

            data_sel = data.loc[np.isin(data[stage_prediction_version], [1,2,3])]
            Xy = function_sel(Xy, data_sel, study_id, name_addon = f'_{stage_v}_NR')

            data_sel = data.loc[np.isin(data[stage_prediction_version], [4])]
            Xy = function_sel(Xy, data_sel, study_id, name_addon = f'_{stage_v}_R')

            data_sel = data.loc[np.isin(data[stage_prediction_version], [5])]
            Xy = function_sel(Xy, data_sel, study_id, name_addon = f'_{stage_v}_W')

    return Xy

In [4]:
def sleep_indices_routine(Xy, data):

    for stage_prediction_version, stage_v in zip(stage_prediction_versions, stage_vs):

        [hours_data_available, hours_sleep, 
                   perc_n1, perc_n2, perc_n3, perc_r, 
                   sfi, sfi2, arousal_index] = sleep_indices(data[stage_prediction_version], verbose=False)

        Xy.loc[Xy.study_id == study_id, 'h_data_' + stage_v] = hours_data_available
        Xy.loc[Xy.study_id == study_id, 'h_sleep_' + stage_v] = hours_sleep
        Xy.loc[Xy.study_id == study_id, 'N1' + stage_v] = perc_n1
        Xy.loc[Xy.study_id == study_id, 'N2' + stage_v] = perc_n2
        Xy.loc[Xy.study_id == study_id, 'N3' + stage_v] = perc_n3
        Xy.loc[Xy.study_id == study_id, 'R' + stage_v] = perc_r
        Xy.loc[Xy.study_id == study_id, 'SFI' + stage_v] = sfi
        Xy.loc[Xy.study_id == study_id, 'SFI2' + stage_v] = sfi2
        Xy.loc[Xy.study_id == study_id, 'ArousalIndex' + stage_v] = arousal_index
    
    return Xy

### let's create a Xy matrix for the icu sleep vs. sleeplab paper. general idea:
inputs: .h5 files with any type of extraced features  
code: compute summary statistics for those features over e.g. a full night for all patients  
output: designmatrix with rows for patients/nights and columns statistics.

In [5]:
# designmatrix initialization

if 0:
    
    # ICU use sleep_stages output table for now as this created a 24hour slicing for all patients:
    sleep_stages_summary = pd.read_csv('/media/mad3/Projects/ICU_SLEEP_STUDY/Sleep_And_Breathing/sleep_staging/sleep_stages_indices_raw_SFI.csv')
    icu_AgeSex = pd.read_csv('/media/mad3/Projects/ICU_SLEEP_STUDY/data/data_analysis/ExportedReports/AgeSex.csv')
    icu_MRN = pd.read_csv('/media/mad3/Projects/ICU_SLEEP_STUDY/data/data_analysis/ExportedReports/LIST.csv')
    icu_AgeSex['study_id'] = icu_AgeSex['Study ID:'].apply(lambda x: 'icu_'+str(x).zfill(3))
    icu_MRN['study_id'] = icu_MRN['Study ID:'].apply(lambda x: 'icu_'+str(x).zfill(3))

    # SLEEPLAB:
    masterlist = '/media/ssd2/Dropbox (Partners HealthCare)/AirGoSleepLabData/MasterList_Main_1.23.20.csv'
    sleeplab_airgo = pd.read_csv(masterlist)
    sleeplab_airgo_demographics = pd.read_csv('sleeplab_airgo_patients_demographics.csv')
    sleeplab_airgo_demographics['study_id'] = sleeplab_airgo_demographics['SID'].apply(lambda x: 'sleeplab_' + x[3:])

    sleeplab_airgo = sleeplab_airgo[sleeplab_airgo.IncludedInStudy==1]
    sleeplab_airgo.loc[sleeplab_airgo.MRN == 1227628,'MRN'] = 1227638
    sleeplab_airgo.loc[sleeplab_airgo.MRN == 1619692,'MRN'] = 1619622
    sleeplab_airgo.loc[sleeplab_airgo.MRN == 1929449,'MRN'] = 2929449
    sleeplab_airgo.loc[sleeplab_airgo.MRN == 4139228,'MRN'] = 4130228
    sleeplab_airgo.loc[sleeplab_airgo.MRN == 4367223,'MRN'] = 4367233
    sleeplab_airgo.loc[sleeplab_airgo.MRN == 5257798,'MRN'] = 5257789
    sleeplab_airgo.loc[sleeplab_airgo.MRN == 6595920,'MRN'] = 6595320
    sleeplab_airgo.loc[sleeplab_airgo.MRN == 34999992,'MRN'] = 2489996
    
    Xy = pd.DataFrame()

    study_id_icu = sleep_stages_summary.Study_ID.apply(lambda x: 'icu_' + str(x).zfill(3))
    night_id_icu = sleep_stages_summary.Start.apply(lambda x: x[:10])

    study_id_sleeplab = sleeplab_airgo.SID.apply(lambda x: 'sleeplab_' + x.replace('Air',''))
    night_id_sleeplab = sleeplab_airgo.Date
    
    
    Xy['study_id'] = np.concatenate([study_id_icu.values, study_id_sleeplab.values], axis=0)
    Xy['night_id'] = np.concatenate([night_id_icu.values, night_id_sleeplab.values], axis=0)
    Xy['population'] = np.concatenate([['icu']*len(study_id_icu), ['sleeplab']*len(study_id_sleeplab)])
    Xy = Xy.reset_index(drop=True)
    
    
    Xy['MRN'] = np.nan
    Xy['Age'] = np.nan
    Xy['Sex'] = np.nan

    Xy.loc[Xy.population=='sleeplab', 'MRN'] = sleeplab_airgo.MRN.values
    for i, row in Xy[Xy.population=='icu'].iterrows():
        Xy.loc[i, 'MRN'] = icu_MRN[icu_MRN.study_id == row.study_id]['MRN:'].values[0]
        Xy.loc[i, 'Age'] = icu_AgeSex[icu_AgeSex.study_id == row.study_id]['Age'].values[0]
        Xy.loc[i, 'Sex'] = icu_AgeSex[icu_AgeSex.study_id == row.study_id]['Gender'].values[0]

    age_sleeplab = []
    sex_sleeplab = []
    for i, row in Xy[Xy.population=='sleeplab'].iterrows():
        demographics = sleeplab_airgo_demographics[sleeplab_airgo_demographics.MRN.astype(int) == int(row.MRN)]
        Xy.loc[i, 'Age'] = np.round(demographics.Age.values[0],1)
        Xy.loc[i, 'Sex'] = demographics.Sex.values[0]
        
    Xy.to_csv('icu_sleeplab_Xy.csv', index=False)
    
else:
    Xy = pd.read_csv('icu_sleeplab_Xy.csv')
    
    
variables_in_x = ['IBI_mean', 'IBI_std', 'IBI_median', 'IBI_Q75', 'IBI_Q25', 'IBI_RMSSD', 'IBI_SDA', 'IBI_ASD']
for variable in variables_in_x:
    if not variable in Xy.columns:
        Xy[variable] = np.nan

In [6]:
Xy.head(2)

,study_id,night_id,population,MRN,Age,Sex,IBI_mean,IBI_std,IBI_median,IBI_Q75,...,ArousalIndexSSComb1,h_data_SSExpert,h_sleep_SSExpert,N1SSExpert,N2SSExpert,N3SSExpert,RSSExpert,SFISSExpert,SFI2SSExpert,ArousalIndexSSExpert
0,icu_001,2018-06-05,icu,1298742.0,79.0,Male,7.958415,5.441516,5.5,15.0,...,7714.3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,icu_001,2018-06-06,icu,1298742.0,79.0,Male,7.958415,5.441516,5.5,15.0,...,7714.3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


#### breathing summary statistics

In [7]:
import time
a = time.time()

####################### params ################################

cohort = 'sleeplab'

data_dir = f'/media/ssd2/{cohort}_final_files'

# Define what Sleep Stage version shall be used:
stage_prediction_versions = ['stage_pred_amendsumeffort', 'stage_pred_vcomb1']   # actual column names in data
stage_vs = ['SSResp', 'SSComb1']                                                  # short versions for variable names in Xy
if cohort == 'sleeplab':
    stage_prediction_versions += ['stage']
    stage_vs += ['SSExpert']
    
####################### code ###################################

files = os.listdir(data_dir)
print(len(files))

for file in tqdm(files):

    compute_existing_data_again = False
    if not compute_existing_data_again:
        signals_contained, hdr = get_metadata(os.path.join(data_dir, file))
        study_id = cohort + '_' + str(hdr['study_id']).zfill(3)
        # if there is already a valid value contained for this study_id, skip it:
        if not pd.isna(Xy.loc[Xy.study_id == study_id, 'IBI_mean'].iloc[0]):
            continue


    data, hdr = load_sleep_data(os.path.join(data_dir, file), idx_to_datetime=1)
    study_id = cohort + '_' + str(hdr['study_id']).zfill(3)
    fs = hdr['fs']
    
    # compute breathing statistics and add to Xy:
    Xy = breathing_metrics_routine(Xy, data)
    
    # compute sleep indices and add to Xy:
    Xy =  sleep_indices_routine(Xy, data)

    Xy.to_csv('icu_sleeplab_Xy.csv', index=False)
    

  2%|▏         | 9/412 [00:00<00:04, 83.94it/s]

412


100%|██████████| 412/412 [00:04<00:00, 88.66it/s] 


In [7]:
####################### params ################################

cohort = 'icu'

data_dir = f'/media/ssd2/{cohort}_final_files'

# Define what Sleep Stage version shall be used:
stage_prediction_versions = ['stage_pred_amendsumeffort', 'stage_pred_vcomb1']   # actual column names in data
stage_vs = ['SSResp', 'SSComb1']                                                  # short versions for variable names in Xy
if cohort == 'sleeplab':
    stage_prediction_versions += ['stage']
    stage_vs += ['SSExpert']
    
####################### code ###################################

files = os.listdir(data_dir)
print(len(files))

for file in tqdm(files):

    compute_existing_data_again = False
    if not compute_existing_data_again:
        signals_contained, hdr = get_metadata(os.path.join(data_dir, file))
        study_id = cohort + '_' + str(hdr['study_id']).zfill(3)
        # if there is already a valid value contained for this study_id, skip it:
        if not pd.isna(Xy.loc[Xy.study_id == study_id, 'IBI_mean'].iloc[0]):
            print('skip')
            continue
            
    print('do')
            
            
    data, hdr = load_sleep_data(os.path.join(data_dir, file), idx_to_datetime=1)
    study_id = cohort + '_' + str(hdr['study_id']).zfill(3)
    fs = hdr['fs']
    
    # compute breathing statistics and add to Xy:
    Xy = breathing_metrics_routine(Xy, data)
    
    # compute sleep indices and add to Xy:
    Xy =  sleep_indices_routine(Xy, data)

    Xy.to_csv('icu_sleeplab_Xy.csv', index=False)
    

  9%|▉         | 12/132 [00:00<00:01, 116.93it/s]

132
skip
skip
skip
skip
skip
skip
skip
skip
skip
skip
skip
skip
skip
skip
skip
skip
skip
skip
skip
skip
skip
skip
skip
skip

 30%|██▉       | 39/132 [00:00<00:00, 122.06it/s]


skip
skip
skip
skip
skip
skip
skip
skip
skip
skip
skip
skip
skip
skip
skip
do


 30%|██▉       | 39/132 [00:20<00:00, 122.06it/s]/home/wolfgang/anaconda3/envs/AirGoSleep/lib/python3.6/site-packages/numpy/core/fromnumeric.py:3257: RuntimeWarning: Mean of empty slice.
  out=out, **kwargs)
/home/wolfgang/anaconda3/envs/AirGoSleep/lib/python3.6/site-packages/numpy/core/_methods.py:161: RuntimeWarning: invalid value encountered in true_divide
  ret = ret.dtype.type(ret / rcount)
 30%|███       | 40/132 [00:36<16:28, 10.74s/it] 

...

skip
skip
skip
skip
do


 79%|███████▉  | 104/132 [04:49<07:25, 15.90s/it]

do


100%|██████████| 132/132 [05:25<00:00,  2.47s/it]

skip
skip
skip
skip
skip
skip
skip
skip
skip
skip
skip
skip
skip
skip
skip
skip
skip
skip
skip
skip
skip
skip
skip
skip
skip
skip
skip


In [11]:
b = time.time()-a
print(b/60)

103.95474268595378


In [31]:
Xy.loc[Xy.study_id == study_id]

,study_id,night_id,population,MRN,Age,Sex,IBI_mean,IBI_std,IBI_median,IBI_Q75,...,ArousalIndexSSComb1,h_data_SSExpert,h_sleep_SSExpert,N1SSExpert,N2SSExpert,N3SSExpert,RSSExpert,SFISSExpert,SFI2SSExpert,ArousalIndexSSExpert
609,sleeplab_237,2019-04-23,sleeplab,5238596.0,48.7,Female,NaN,NaN,NaN,NaN,...,7.4,7.3,6.875,2.0,60.0,21.0,16.0,1.6,0.6,0.9
